# Homework 1: Non-linear Programming
## Shaun Ketner

More on the linear assignment problem. A barrage of 1000 missiles is incoming.
You have 1000 interceptors and need to assign them to the incoming missiles. The matrix
C PR1000ˆ1000 is such that Cij gives the initial distance between missile i and interceptor j.
Find this matrix at https://faculty.nps.edu/rbassett/Data/DistanceMatrix.txt. It’s a text
file so load it in with numpy.loadtxt. Find an assignment of interceptors to missiles that
minimizes the cumulative sum of distances each interceptor must travel.

In [1]:
import numpy as np
import cvxpy as cvx
import scipy.sparse as sparse
import time

## 1. Formulate and solve this problem as you would in Pyomo.

In [3]:
n = 1000
C = np.loadtxt('DistanceMatrix.txt')

In [4]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()
missiles=range(n)
interceptors=range(n)
#Nonnegative flow variables
model.X = pyo.Var(missiles, interceptors, domain=pyo.NonNegativeReals)
#Parameters
def cost(model, i, j):
    return float(C[i,j])
model.c=pyo.Param(missiles, interceptors, rule=cost)

#objective Function
def missileCost(model):
    return sum(model.c[i,j]*model.X[i,j] for i in missiles for j in interceptors)   #objective function is the sum of the cost of each missile times each missile assigned to an interceptor
model.MissileCost = pyo.Objective(rule=missileCost, sense=pyo.minimize) # minimize the objective function

#balance or flow Constraints
def incoming_missile_rule(model,j):  
    return sum(model.X[i,j] for i in missiles) == 1 
model.missile_capacity = pyo.Constraint(missiles, rule=incoming_missile_rule)

def interceptor_rule(model,i): 
    return sum(model.X[i,j] for j in interceptors) == 1 
model.interceptor_capacity = pyo.Constraint(interceptors, rule=interceptor_rule)

opt=pyo.SolverFactory('appsi_highs')
start = time.time()
result=opt.solve(model)
print("Model Results", result)
print(f"Optimal objective: {pyo.value(model.MissileCost)}")
print(f"Took {time.time() - start} seconds to compile and solve")

Model Results 
Problem: 
- Lower bound: 263.87881363906627
  Upper bound: 263.87881363906627
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
Solution: 
- number of solutions: 0
  number of solutions displayed: 0

Optimal objective: 263.87881363906627
Took 23.920682191848755 seconds to compile and solve


## 2. Formulate and solve this problem using at least some scalar notation in CVXPY.1

In [5]:
#second attempt. Something between matrix and scalar form
X = cvx.Variable((n,n), nonneg=True)
objective = cvx.Minimize(cvx.sum(cvx.multiply(C, X)))
#first we'll do row column sums
constraints = [cvx.sum(X[:,j]) == 1. for j in range(n)]
#now we'll do row sums
constraints += [cvx.sum(X[i,:]) == 1. for i in range(n)]
#finally we'll add the upper bounds to the variables
constraints += [X <= 1.]
prob2 = cvx.Problem(objective, constraints)
start = time.time()
prob2.solve(solver='HIGHS', verbose=True)
print(f"Optimal value: {prob2.value:.6f}")
print(f"Took {time.time() - start} seconds to compile and solve")

(CVXPY) Jul 13 08:39:21 PM: Your problem has 1000000 variables, 1002000 constraints, and 0 parameters.
(CVXPY) Jul 13 08:39:21 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 13 08:39:21 PM: DCP verification time: 0.0834 seconds.
(CVXPY) Jul 13 08:39:21 PM: Expression tree has 3 nodes.
(CVXPY) Jul 13 08:39:21 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 13 08:39:21 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 13 08:39:21 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 13 08:39:21 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 13 08:39:21 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 13 08:39:21 PM: Applying reduction Dcp2Cone


                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 13 08:39:21 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jul 13 08:39:21 PM: Applying reduction EliminateZeroSized
(CVXPY) Jul 13 08:39:21 PM: Applying reduction ConeMatrixStuffing
(CVXPY) Jul 13 08:41:41 PM: Applying reduction HIGHS
(CVXPY) Jul 13 08:41:43 PM: Finished problem compilation (took 1.420e+02 seconds).
(CVXPY) Jul 13 08:41:43 PM: Invoking solver HIGHS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------


(CVXPY) Jul 13 08:41:54 PM: Problem status: optimal
(CVXPY) Jul 13 08:41:54 PM: Optimal value: 2.639e+02
(CVXPY) Jul 13 08:41:54 PM: Compilation took 1.420e+02 seconds
(CVXPY) Jul 13 08:41:54 PM: Solver (including time spent in interface) took 1.138e+01 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Optimal value: 263.878814
Took 153.57699513435364 seconds to compile and solve


## 3. Formulate and solve this problem using matrix/vector notation in CVXPY. Use sparse matrices where appropriate.

In [7]:
#another attempt. This time use bounds instead of x <= 1. constraint
c = C.ravel()
x = cvx.Variable(n*n, bounds=(0., 1.))
objective = cvx.Minimize(c.T @ x)
#sum over interceptors matrix
sum_int_mat = sparse.coo_array((n, n*n))
for i in range(n):
    sum_int_mat[i,n*i:n*(i+1)] = 1.
#sum over missiles matrix
sum_miss_mat = sparse.coo_array((n, n*n))
for i in range(n):
    sum_miss_mat[i,i::n] = 1.
D = sparse.bmat([[sum_int_mat],[sum_miss_mat]])
constraints = [D @ x == 1.]
prob3 = cvx.Problem(objective, constraints)
start = time.time()
prob3.solve(solver='HIGHS', verbose=True)
print(f"Optimal value: {prob3.value:.6f}")
print(f"Took {time.time() - start} seconds to compile and solve")


(CVXPY) Jul 13 08:42:43 PM: Your problem has 1000000 variables, 2000 constraints, and 0 parameters.
(CVXPY) Jul 13 08:42:43 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 13 08:42:43 PM: DCP verification time: 0.0001 seconds.
(CVXPY) Jul 13 08:42:43 PM: Expression tree has 1 nodes.
(CVXPY) Jul 13 08:42:43 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 13 08:42:43 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 13 08:42:43 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 13 08:42:43 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 13 08:42:43 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 13 08:42:43 PM: Applying reduction Dcp2Cone
(CVXPY) Jul 13 08:42:43 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jul 13 08:42:43 PM: 

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 13 08:42:44 PM: Applying reduction HIGHS
(CVXPY) Jul 13 08:42:44 PM: Finished problem compilation (took 1.653e+00 seconds).
(CVXPY) Jul 13 08:42:44 PM: Invoking solver HIGHS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------


(CVXPY) Jul 13 08:42:53 PM: Problem status: optimal
(CVXPY) Jul 13 08:42:53 PM: Optimal value: 2.639e+02
(CVXPY) Jul 13 08:42:53 PM: Compilation took 1.653e+00 seconds
(CVXPY) Jul 13 08:42:53 PM: Solver (including time spent in interface) took 8.765e+00 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Optimal value: 263.878814
Took 10.541803121566772 seconds to compile and solve


## 4. Compute the dual. Reuse the matrices and vectors defined in the previous problem to
solve the dual problem in CVXPY. Confirm that strong duality holds.

In [9]:
y = cvx.Variable(n*n, nonneg=True)
z = cvx.Variable(2*n)
dual_obj = cvx.Maximize(-cvx.sum(y)-cvx.sum(z))
dual_cons=[c+y+D.T@z>=0]
dual_prob = cvx.Problem(dual_obj, dual_cons)
start = time.time()
dual_prob.solve(solver='HIGHS', verbose=True)
print(f"Primal Objective value: {prob3.value}")
print(f"Dual objective value: {dual_prob.value}")
print(f"Took {time.time() - start} seconds to compile and solve")

(CVXPY) Jul 13 08:48:58 PM: Your problem has 1002000 variables, 1000000 constraints, and 0 parameters.
(CVXPY) Jul 13 08:48:58 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 13 08:48:58 PM: DCP verification time: 0.0002 seconds.
(CVXPY) Jul 13 08:48:58 PM: Expression tree has 4 nodes.
(CVXPY) Jul 13 08:48:58 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 13 08:48:58 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 13 08:48:58 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 13 08:48:58 PM: Compiling problem (target solver=HIGHS).
(CVXPY) Jul 13 08:48:58 PM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> HIGHS
(CVXPY) Jul 13 08:48:58 PM: Applying reduction FlipObjective
(CVXPY) Jul 13 08:48:58 PM: Applying reduction Dcp2Cone
(CVXPY) J

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 13 08:48:58 PM: Applying reduction HIGHS
(CVXPY) Jul 13 08:48:59 PM: Finished problem compilation (took 1.625e+00 seconds).
(CVXPY) Jul 13 08:48:59 PM: Invoking solver HIGHS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------


(CVXPY) Jul 13 08:53:50 PM: Problem status: optimal
(CVXPY) Jul 13 08:53:50 PM: Optimal value: 2.639e+02
(CVXPY) Jul 13 08:53:50 PM: Compilation took 1.625e+00 seconds
(CVXPY) Jul 13 08:53:50 PM: Solver (including time spent in interface) took 2.910e+02 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Primal Objective value: 263.87881363906627
Dual objective value: 263.8788136390664
Took 292.7608253955841 seconds to compile and solve
